# Legumin Molecular Dynamics Pipeline

**Protein:** pea legumin homodimer (UniProt P02857): two chains, 517 residues and 8,149 atoms each after `pdb2gmx` (1,034 residues, 16,298 atoms in total)
**Pipeline:** AlphaFold3 structure → GROMACS MD → trajectory analysis → PyMOL visualization
**Conditions:** 300 K and 400 K, each analysed over a 200 ps and a 500 ps window (the 200 ps window is the first 200 ps of the 500 ps trajectory)
**Environment:** Google Colab, Tesla T4; GROMACS 2026.3 (conda-forge CUDA build, binary `gmx_mpi`); AMBER99SB-ILDN force field; SPC/E water; 0.15 M NaCl
**System (from the original run):** cubic box 18.94 nm, 669,214 atoms = 16,298 protein + 651,678 water atoms (217,226 molecules) + 1,238 ions

---

### Read this first: provenance and known caveats

1. **Assembled, not re-executed.** This notebook was assembled from several working sessions and cleaned. It has not been run end-to-end in exactly this form. Commands were taken from the original session outputs; places where this notebook differs from what was originally typed are marked **[changed]**.
2. **Equilibration (section 4) is reconstructed.** The original NVT/NPT commands were not preserved. The `.mdp` files and the logs (`nvt*.log`, `npt*.log`, all GROMACS 2026.3) exist; the commands follow the standard pattern.
3. **GROMACS versions in the original runs.** Everything ran on 2026.3 **except the first 200 ps of the 400 K production run, which ran on 2024.5**. It was continued from its checkpoint (step 100,000 = 200 ps) with 2026.3, and GROMACS warned that cross-version checkpoint continuation is not officially supported. A re-run with one version will not be bit-identical. A check for a discontinuity at the restart is in section 8 (none was found in Rg, SASA or H-bonds).
4. **Energy minimization did not reach the requested force tolerance** (details in section 3). The system went on to equilibrate and run normally.
5. **300 K vs 400 K production were run differently**: the 300 K extension used `-noappend` + `trjcat`; the 400 K extension appended to the original files (section 5).
6. **Legumin and vicilin setups are not identical.** The legumin runs use Parrinello-Rahman pressure coupling and no explicit dispersion correction; the vicilin runs use C-rescale and `DispCorr = EnerPres`. Cross-protein comparisons are qualitative. Legumin's `nvt_300K.mdp` also differs from the other NVT files (force-switched vdW, no position-restraint define).
7. **Single trajectory per condition, 200-500 ps.** Results are hypothesis-generating; no replicates.
8. **Not in this notebook:** AlphaFold3 structure prediction (AlphaFold Server) and the confidence/chain-coloured prediction images (rendered in PyMOL on a local machine, see `alphafold_predictions/`).

**Runtime reference (Tesla T4):** the 300 ps extension of the 300 K run took about 53 min wall time (about 8.1 ns/day).

### File-name map (original session → this notebook)

| Original session | This notebook |
|---|---|
| `md.*` (300 K, 0-200 ps) | `md_300K_200ps.*` |
| `md_300k_500ps_extended.part000N.*` | `md_300K_ext.part*.*` |
| `md_300k_full_500ps.xtc` | `md_300K_500ps.xtc` |
| `production3.*` (400 K) | `md_400K_200ps.*` (appended to 500 ps) |
| `production3_500ps.tpr` | `md_400K_500ps.tpr` |
| `final_300K.*`, `final_400K.*` | unchanged |

## 1. Environment setup

`condacolab.install()` **restarts the runtime** automatically. After it restarts, continue from the next cell (do not re-run this one).

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.TARGET_PYTHON = "3.13"   # as used in the original working session
condacolab.install()

In [ ]:
# CUDA build of GROMACS 2026.3 (conda-forge). This installs the binary `gmx_mpi`.
# NOTE: plain `gromacs=2026.3` gave an OpenCL (non-CUDA) build in the original session, and
# `gromacs=*=*cuda*` (no version) resolved to 2024.5, so version AND build are pinned here.
!mamba install -y -c conda-forge -c bioconda "gromacs=2026.3=mpi_openmpi_cuda*"
# If the solver picks a non-CUDA build, use the exact build that worked originally:
# !mamba install -y -c conda-forge -c bioconda "gromacs=2026.3=mpi_openmpi_cuda_h56f47eb_0"

In [ ]:
# [changed] The original session called `gmx_mpi`; a symlink lets every command below use `gmx`.
!ln -sf $(which gmx_mpi) /usr/local/bin/gmx
!gmx --version | grep -E "GROMACS version|GPU support|MPI library|CUDA (driver|runtime)"
!nvidia-smi --query-gpu=name,driver_version --format=csv
# Expected: GROMACS version 2026.3-conda_forge, GPU support: CUDA, GPU: Tesla T4
!gmx --version | grep -q "GPU support: *CUDA" || echo "WARNING: this is not a CUDA build of GROMACS"

In [ ]:
import importlib
for m in ["numpy", "matplotlib", "scipy", "pandas", "PIL"]:
    try:
        importlib.import_module(m)
    except ImportError:
        print("missing:", m, "-> run: !pip install", "pillow" if m == "PIL" else m)
print("python libraries checked")

### Input files

The AlphaFold3 dimer (`structures/legumin_dimer.pdb`) and all `.mdp` files come from the repository. Always run `mdrun` from the local Colab disk (`/content/run`), not from the Drive mount. Drive is only used for backups (optional).

In [ ]:
import os
os.makedirs("/content/run", exist_ok=True)
!git clone --depth 1 https://github.com/malikahmedhassanawan/Legumin-Vicilin-Comparative-MD /content/repo
!cp /content/repo/structures/legumin_dimer.pdb /content/run/legumin_legumin.pdb
!cp /content/repo/simulation_setup/legumin/*.mdp /content/run/
os.chdir("/content/run")
!ls

In [ ]:
# Optional: back up results to Drive after each stage (Colab sessions can end without warning).
from google.colab import drive
drive.mount("/content/drive")
import glob, shutil
BACKUP = "/content/drive/MyDrive/Legumin_MD_backup"

def backup(patterns=("*.xtc", "*.cpt", "*.tpr", "*.log", "*.gro", "*.edr", "*.xvg", "*.dat", "*.top", "*.ndx")):
    os.makedirs(BACKUP, exist_ok=True)
    for p in patterns:
        for f in glob.glob(p):
            shutil.copy2(f, BACKUP)
    print("backed up to", BACKUP)

## 2. System preparation

Starting structure: the AlphaFold3-predicted homodimer (`legumin_legumin.pdb` = `structures/legumin_dimer.pdb`).

In [ ]:
# Topology: AMBER99SB-ILDN force field, SPC/E water, hydrogens rebuilt (-ignh)
!gmx pdb2gmx -f legumin_legumin.pdb -o legumin_processed.gro -water spce -ff amber99sb-ildn -ignh
# Expected (original run): 2 chains x (517 residues, 8,149 atoms) = 16,298 atoms, 1,034 residues;
# disulfide bonds detected automatically; total charge -10 e; total mass 117,594 u.

In [ ]:
# Cubic box with 1.0 nm padding
!gmx editconf -f legumin_processed.gro -o legumin_box.gro -c -d 1.0 -bt cubic
# Expected: solute diameter 16.94 nm -> cubic box 18.94 nm (volume 6,794 nm^3).
# The box is large because the dimer is extended, so about 97% of all atoms end up as solvent.

In [ ]:
!gmx solvate -cp legumin_box.gro -cs spc216.gro -o legumin_solv.gro -p topol.top
# Expected: 218,464 water molecules; 671,690 atoms; density about 993 g/l

In [ ]:
# Neutralize and add 0.15 M NaCl. The system charge is -10 e before ions, hence -maxwarn 1.
!gmx grompp -f ions.mdp -c legumin_solv.gro -p topol.top -o ions.tpr -maxwarn 1
# [changed] The solvent group is chosen explicitly (it is group 13 "SOL" in this system).
!echo "SOL" | gmx genion -s ions.tpr -o legumin_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15
# Expected: 1,238 ions replace 1,238 water molecules -> 217,226 waters, 669,214 atoms in total.

In [ ]:
# Sanity check: [ molecules ] must list both protein chains, SOL, NA and CL.
# (In one original session topol.top was overwritten by a protein-only version, which breaks grompp.)
!tail -8 topol.top

## 3. Energy minimization

`em.mdp`: steepest descent, `emtol` 1000 kJ/mol/nm, up to 50,000 steps, PME, 1.0 nm cutoffs.

**Original result:** steepest descent stopped after 2,781 steps because it reached machine precision, **without reaching Fmax < 1000** (Epot = -1.2387 x 10^7 kJ/mol; Fmax = 7.1 x 10^3 kJ/mol/nm on atom 8619). The run was continued into equilibration regardless; both NVT and NPT completed without errors.

In [ ]:
!gmx grompp -f em.mdp -c legumin_ions.gro -p topol.top -o em.tpr -maxwarn 1
!gmx mdrun -v -deffnm em -nb gpu
!grep -E "Potential Energy|Maximum force|converged" em.log

## 4. NVT and NPT equilibration

> **Reconstructed section.** The original equilibration commands were not preserved. This follows the standard GROMACS pattern with the `.mdp` files in `simulation_setup/legumin/`. The original logs (`nvt.log`, `npt.log`, `nvt_400K.log`, `npt_400K.log`) all report GROMACS 2026.3, and the 300 K NPT checkpoint used for production was written at t = 100 ps.
>
> `nvt_300K.mdp` differs from `nvt_400K.mdp` (force-switched vdW, no position-restraint define), so the `-r` reference in the 300 K NVT step has no restraining effect.

In [ ]:
# NVT (temperature coupling), 300 K and 400 K
!gmx grompp -f nvt_300K.mdp -c em.gro -r em.gro -p topol.top -o nvt_300K.tpr -maxwarn 1
!gmx mdrun -v -deffnm nvt_300K -nb gpu
!gmx grompp -f nvt_400K.mdp -c em.gro -r em.gro -p topol.top -o nvt_400K.tpr -maxwarn 1
!gmx mdrun -v -deffnm nvt_400K -nb gpu

In [ ]:
# NPT (pressure coupling), 300 K and 400 K
!gmx grompp -f npt_300K.mdp -c nvt_300K.gro -t nvt_300K.cpt -r nvt_300K.gro -p topol.top -o npt_300K.tpr -maxwarn 1
!gmx mdrun -v -deffnm npt_300K -nb gpu
!gmx grompp -f npt_400K.mdp -c nvt_400K.gro -t nvt_400K.cpt -r nvt_400K.gro -p topol.top -o npt_400K.tpr -maxwarn 1
!gmx mdrun -v -deffnm npt_400K -nb gpu
backup()

## 5. Production MD

Time step 2 fs; 200 ps = 100,000 steps; 500 ps = 250,000 steps. Compressed coordinates are written every 500 steps (1 ps): 201 frames after 200 ps, 501 after 500 ps. Each temperature was run to 200 ps and then extended by 300 ps with `gmx convert-tpr -extend 300`.

Key settings in the original `md.mdp` (the `.mdp` files in `simulation_setup/legumin/` are authoritative): LINCS constraints on h-bonds, PME electrostatics, 1.0 nm cutoffs, V-rescale thermostat (tau_t 0.1 ps, Protein / Non-Protein groups), Parrinello-Rahman barostat (tau_p 2.0 ps, 1 bar, isotropic, compressibility 4.5 x 10^-5 bar^-1). The `md_*_500ps.mdp` files are kept for reference; the 500 ps runs were produced by extension.

### 5a. 300 K

The original 300 K extension used `-noappend`, which writes a new set of `*.part000N.*` files that are then joined with `gmx trjcat`. (The original run produced `part0004` because earlier attempts had already used parts 2-3; a fresh run produces `part0002`.)

In [ ]:
# 200 ps production
!gmx grompp -f md_300K_200ps.mdp -c npt_300K.gro -t npt_300K.cpt -p topol.top -o md_300K_200ps.tpr
!gmx mdrun -v -deffnm md_300K_200ps -nb gpu
backup()

In [ ]:
# Extend to 500 ps (about 53 min on a T4), keep part files, then join with the first 200 ps
!gmx convert-tpr -s md_300K_200ps.tpr -extend 300 -o md_300K_500ps.tpr
!gmx mdrun -v -deffnm md_300K_ext -s md_300K_500ps.tpr -cpi md_300K_200ps.cpt -nb gpu -noappend
!gmx trjcat -f md_300K_200ps.xtc $(ls md_300K_ext.part*.xtc) -o md_300K_500ps.xtc
!gmx check -f md_300K_500ps.xtc 2>&1 | grep -E "Last frame|# Atoms"
# Expected: 669214 atoms, last frame at 500 ps
backup()

### 5b. 400 K

The original 400 K run was extended **in place** (checkpoint `-cpi`, same `-deffnm`), so the 500 ps trajectory ends up in the original file names.

> In the original run the first 200 ps used GROMACS 2024.5 and the extension used 2026.3 (see the caveats at the top). Run with a single version here, the result will be statistically comparable but not identical.

In [ ]:
# 200 ps production
!gmx grompp -f md_400K_200ps.mdp -c npt_400K.gro -t npt_400K.cpt -p topol.top -o md_400K_200ps.tpr
!gmx mdrun -v -deffnm md_400K_200ps -nb gpu
backup()

In [ ]:
# Extend to 500 ps, appending to the existing files
!gmx convert-tpr -s md_400K_200ps.tpr -extend 300 -o md_400K_500ps.tpr
!gmx mdrun -v -deffnm md_400K_200ps -s md_400K_500ps.tpr -cpi md_400K_200ps.cpt -nb gpu
!cp md_400K_200ps.xtc md_400K_500ps.xtc
!gmx check -f md_400K_500ps.xtc 2>&1 | grep -E "Last frame|# Atoms"
backup()

In [ ]:
# Which GROMACS version wrote each log? (a log with two version lines was restarted on another version)
!grep -H "GROMACS version" *.log

## 6. Trajectory processing

Two processed versions of each 500 ps trajectory are made:

- `md_<T>_500ps_center.xtc`: whole system, molecules made whole and centred on the protein, used for RMSD, Rg, SASA, H-bonds and RMSF.
- `final_<T>.xtc` / `final_<T>.pdb`: protein only, centred and then fitted (rotation + translation), used for DSSP and the PyMOL movies. The PDB (frame 0) carries the chain IDs.

Group numbers in this system: 0 System, 1 Protein, 3 C-alpha, 4 Backbone.

> The original 300 K centring did not use `-ur compact`, the 400 K one did. The same flags are used for both here; the flag changes only how atoms are placed in the unit cell.

In [ ]:
import subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print(r.stderr[-1500:])
    return r

TEMPS = ["300K", "400K"]
WINDOWS = {"200ps": 200, "500ps": 500}   # analysis windows (ps), both start at 0

for T in TEMPS:
    tpr, xtc = f"md_{T}_500ps.tpr", f"md_{T}_500ps.xtc"
    sh(f'echo "1 0" | gmx trjconv -s {tpr} -f {xtc} -o md_{T}_500ps_center.xtc -pbc mol -center -ur compact')
    sh(f'printf "1\\n1\\n" | gmx trjconv -s {tpr} -f {xtc} -o step1_{T}.xtc -pbc mol -center')
    sh(f'printf "1\\n1\\n" | gmx trjconv -s {tpr} -f step1_{T}.xtc -o final_{T}.xtc -fit rot+trans')
    sh(f'echo "1" | gmx trjconv -s {tpr} -f final_{T}.xtc -o final_{T}.pdb -dump 0')
    print(T, "processed")
backup()

## 7. Analysis

Whole-dimer metrics: RMSD of the backbone (fit and output group 4), radius of gyration and SASA of the protein (group 1), **protein-protein hydrogen bonds** (reference and target both group 1, so bonds within and between the two chains), and per-residue RMSF of the backbone. Time is written in **ps** for every metric.

> **[changed] Time unit.** The earlier clean notebook wrote RMSD in ns but `gyrate` always writes ps, and then labelled both plots "ns". Everything is ps here. (Some files in the repository were written in ns and some in ps; the statistics scripts below detect this automatically.)

The 200 ps window is the first 200 ps of the 500 ps trajectory (`-b 0 -e 200`).

In [ ]:
for T in TEMPS:
    tpr, ctr = f"md_{T}_500ps.tpr", f"md_{T}_500ps_center.xtc"
    for W, end in WINDOWS.items():
        c, opt = f"{T}_{W}", f"-b 0 -e {end}"
        sh(f'echo "4 4" | gmx rms -s {tpr} -f {ctr} -o rmsd_{c}.xvg -tu ps {opt}')
        sh(f'echo "1" | gmx gyrate -s {tpr} -f {ctr} -o rg_{c}.xvg {opt}')
        sh(f'echo "1" | gmx sasa -s {tpr} -f {ctr} -o sasa_{c}.xvg {opt}')
        sh(f'printf "1\\n1\\n" | gmx hbond -s {tpr} -f {ctr} -num hbonds_{c}.xvg {opt}')
        sh(f'echo "4" | gmx rmsf -s {tpr} -f {ctr} -o rmsf_{c}.xvg -res {opt}')
        print("done", c)
backup()

### Per-chain RMSF

Chain A = atoms 1-8,149 and chain B = atoms 8,150-16,298 (each chain has 8,149 atoms after `pdb2gmx`).

> **[changed] Chain groups.** `make_ndx` on a `.tpr` with `chain A` / `chain B` finds 0 atoms, because a `.tpr` does not store chain IDs. The original sessions built the groups from a PDB that carries chain IDs. Atom ranges are used here instead, which gives the same two groups.
>
> **Compare like with like:** the per-chain RMSF files use all protein atoms of each chain (as in the original sessions); the combined RMSF above uses the backbone only.

In [ ]:
for T in TEMPS:
    tpr, ctr = f"md_{T}_500ps.tpr", f"md_{T}_500ps_center.xtc"
    sh(f'printf "a 1-8149\\nname 17 chA\\na 8150-16298\\nname 18 chB\\nq\\n" | gmx make_ndx -f {tpr} -o index_chains_{T}.ndx')
    for W, end in WINDOWS.items():
        for ch in ["chA", "chB"]:
            sh(f'echo "{ch}" | gmx rmsf -s {tpr} -f {ctr} -n index_chains_{T}.ndx -o rmsf_{T}_{W}_{ch}.xvg -res -b 0 -e {end}')
    print(T, "per-chain RMSF done")
backup()

In [ ]:
import numpy as np

def read_xvg(fname):
    data = []
    with open(fname) as f:
        for line in f:
            if line.startswith(("#", "@")):
                continue
            if line.strip():
                data.append([float(x) for x in line.split()])
    return np.array(data)

# Peak RMSF per chain (whole-dimer backbone file: first half = chain A, second half = chain B)
for T in TEMPS:
    r = read_xvg(f"rmsf_{T}_500ps.xvg")
    half = len(r) // 2
    for name, part in [("A", r[:half]), ("B", r[half:])]:
        k = np.argmax(part[:, 1])
        print(f"{T} chain {name}: peak at position {k + 1} within the chain, RMSF = {part[k, 1]:.3f} nm")

### Secondary structure (DSSP)

`gmx dssp` is built into GROMACS (no separate `mkdssp` needed; the original session also installed the apt package `dssp` 4.0.4, which this command does not use). It runs on the protein-only fitted trajectory. The parsed CSVs and the hotspot (residues 260-320) statistics in `analysis/legumin/DSSP/` were produced afterwards with pandas in a separate session and are not part of this notebook.

In [ ]:
for T in TEMPS:
    sh(f'gmx dssp -f final_{T}.xtc -s md_{T}_500ps.tpr -o legumin_{T}_500ps_dssp.dat')
    print(T, "DSSP done")
backup()

## 8. Within-run statistics and restart check

Block statistics for Rg, SASA and H-bonds. The first 25% of each series is discarded as equilibration; 5-block averaging gives the SEM; Δ is the mean of the last 10% minus the mean of the first 10% (relative to the first-10% mean); `delta_over_SD` compares Δ to the run's own fluctuation.

> These are **within-run** errors from one trajectory. Consecutive frames are correlated and a drifting series inflates SD and SEM, so `delta_over_SD` is a descriptive measure, not a significance test. They do not replace independent replicates.

In [ ]:
import pandas as pd

EQUIL, NBLOCKS, EDGE = 0.25, 5, 0.10
out = []
for T in TEMPS:
    for W in WINDOWS:
        c = f"{T}_{W}"
        for m in ["rg", "sasa", "hbonds"]:
            y = read_xvg(f"{m}_{c}.xvg")[:, 1]
            n = max(int(len(y) * EDGE), 2)
            first, last = y[:n].mean(), y[-n:].mean()
            delta = last - first
            yp = y[int(len(y) * EQUIL):]
            bm = np.array([b.mean() for b in np.array_split(yp, NBLOCKS)])
            sem = bm.std(ddof=1) / np.sqrt(NBLOCKS)
            sd = yp.std(ddof=1)
            out.append(dict(protein="legumin", condition=c, metric=m, mean=yp.mean(),
                            SEM_block=sem, SD=sd, delta=delta,
                            delta_pct=100 * delta / first, delta_over_SD=abs(delta) / sd))
df = pd.DataFrame(out).round(4)
df.to_csv("summary_table_stats_legumin.csv", index=False)
print(df.to_string())

### Restart check (400 K, 200 ps)

In the original 400 K run the GROMACS version changed at 200 ps (2024.5 → 2026.3). This prints the change in the 50 ps window average at every boundary; a version-related discontinuity would show as an unusually large step at 200 ps. **Original result:** the step at 200 ps was among the smallest for Rg (+0.56 SD), SASA (+0.45 SD) and H-bonds (-0.02 SD). Most of the Rg decrease occurred after 200 ps, so an effect on the drift rate cannot be excluded by this test.

In [ ]:
W_PS = 50
for name in ["rg", "sasa", "hbonds"]:
    a = read_xvg(f"{name}_400K_500ps.xvg")
    t, y = a[:, 0], a[:, 1]
    if t.max() < 5:          # time written in ns
        t = t * 1000
    edges = np.arange(0, t.max() + W_PS, W_PS)
    means = [y[(t >= e) & (t < e + W_PS)].mean() for e in edges[:-1]]
    sd0 = np.std(y[t < 200], ddof=1)
    print(name)
    for e, s in zip(edges[1:-1], np.diff(means)):
        flag = "   <-- 200 ps (restart in the original run)" if abs(e - 200) < 1e-6 else ""
        print(f"  {e:4.0f} ps: {s:+.3f} ({s / sd0:+.2f} SD){flag}")

## 9. Quick visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for row, T in enumerate(TEMPS):
    rmsd = read_xvg(f"rmsd_{T}_500ps.xvg")
    rg = read_xvg(f"rg_{T}_500ps.xvg")
    axes[row, 0].plot(rmsd[:, 0], rmsd[:, 1])
    axes[row, 0].set_ylabel("Backbone RMSD (nm)")
    axes[row, 0].set_title(f"Legumin RMSD, {T}, 500 ps")
    axes[row, 1].plot(rg[:, 0], rg[:, 1])
    axes[row, 1].set_ylabel("Rg (nm)")
    axes[row, 1].set_title(f"Legumin radius of gyration, {T}, 500 ps")
    for ax in axes[row]:
        ax.axvline(200, ls=":", c="gray")   # boundary of the 200 ps window
for ax in axes[1]:
    ax.set_xlabel("Time (ps)")
plt.tight_layout()
plt.savefig("rmsd_rg_300K_400K_500ps.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Trajectory visualization (PyMOL)

Coloured cartoon (chain A red, chain B blue) with **inter-chain** hydrogen bonds (chain A ↔ chain B, 3.5 Å) as yellow dashes, rendered frame by frame and assembled into an MP4 with `ffmpeg`. The dashes show only bonds between the two chains, whereas the H-bond counts in section 7 include bonds within each chain as well.

> Ray-tracing 501 frames at 1920 x 1080 takes a long time. Test with a few frames first (set `MAX_FRAMES = 10`).

In [ ]:
!pip install pymol-open-source --break-system-packages -q

In [ ]:
import pymol
from pymol import cmd
pymol.finish_launching(["pymol", "-qc"])

MAX_FRAMES = None   # e.g. 10 for a quick test; None = all frames

for T in TEMPS:
    cmd.reinitialize()
    cmd.load(f"final_{T}.pdb", "traj")
    cmd.load_traj(f"final_{T}.xtc", "traj")
    cmd.hide("everything")
    cmd.show("cartoon")
    cmd.color("red", "chain A")
    cmd.color("blue", "chain B")
    cmd.bg_color("white")
    cmd.set("ray_opaque_background", 1)
    cmd.set("antialias", 2)
    cmd.set("dash_width", 2)
    cmd.set("label_size", 0)
    cmd.orient()
    cmd.zoom("all", buffer=-11)
    n_frames = cmd.count_states("traj")
    if MAX_FRAMES:
        n_frames = min(n_frames, MAX_FRAMES)
    for i in range(1, n_frames + 1):
        cmd.frame(i)
        cmd.delete("hbonds")
        cmd.distance("hbonds", "chain A and (donor or acceptor)",
                     "chain B and (donor or acceptor)", mode=2, cutoff=3.5)
        cmd.color("yellow", "hbonds")
        cmd.png(f"frame_{T}_{i:04d}.png", width=1920, height=1080, dpi=150, ray=1)
    print(T, "frames rendered:", n_frames)

In [ ]:
# Assemble frames into videos (ffmpeg is preinstalled in Colab)
for T in TEMPS:
    !ffmpeg -y -framerate 20 -i frame_{T}_%04d.png -vcodec libx264 -crf 28 -preset slow -vf scale=720:-1 -an legumin_{T}_500ps.mp4
backup(("*.mp4",))

## 11. Record the software versions of this run

Run this at the end of a session and keep the output with your results.

In [ ]:
!gmx --version | head -12
!python --version
!conda list gromacs
!pip freeze | grep -iE "^(numpy|pandas|scipy|matplotlib|pillow|pymol)" | tee requirements_legumin.txt

## Limitations

- One trajectory per temperature, 200-500 ps, no replicates; within-run error bars only.
- Homodimer only (not the native hexamer assembly); no experimental anchor.
- Energy minimization stopped at machine precision without reaching Fmax < 1000.
- The original 400 K production run spans GROMACS 2024.5 (0-200 ps) and 2026.3 (200-500 ps); no discontinuity was detected at the restart, but most of the Rg decrease occurs after it.
- Legumin and vicilin used different pressure coupling and dispersion-correction settings, so cross-protein comparisons are qualitative.
- Equilibration commands (section 4) are reconstructed.